<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-06-function-calling/lesson-6.1-function-calling/notebooks/GCP_Capstone_6.1_FunctionCalling.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.1 Gemini Function Calling — FunctionDeclarations, AUTO Mode
**Netsetos GenAI Engineering — GCP Capstone**

Teach Gemini to use tools. The model decides which to call, extracts arguments, you execute.


## Setup


In [ ]:
!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='global')
print(f'Client ready for {PROJECT_ID}')


## Cell 1: Define DocuMind Functions


In [ ]:
# DocuMind's tool layer. Every lesson in Modules 6, 7 and
# 8 uses THIS contract - one name, one shape, one corpus.
USD_INR = 85          # course-wide conversion rate
RATES = {"standard": 0.05, "priority": 0.12, "bulk": 0.03}

# DocuMind's demo corpus - the SAME five documents in every
# lesson of Modules 6, 7 and 8, so results stay comparable.
# 119 pages in total, which is what the cost examples bill.
GS = "gs://documind-acme"
_DOCS = [
    # doc_id, doc_type, page, score, quote
    ("hr_policy_2026", "policy", 12, 0.94,
     "A senior engineer serves a notice period of 60 days."),
    ("hr_policy_2026", "policy", 31, 0.81,
     "Earned leave is encashed on exit, capped at 45 days."),
    ("msa_acme_2026", "contract", 8, 0.88,
     "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "invoice", 1, 0.76,
     "Total payable Rs 1,84,500, inclusive of 18% GST."),
    ("gstr1_q1_fy27", "form", 4, 0.68,
     "Outward taxable supplies for the quarter, GSTR-1."),
    ("rag_survey_2026", "research_paper", 6, 0.72,
     "Hybrid retrieval mixes dense and sparse signals."),
]
CORPUS = [{"chunk_id": f"{d}#{p}", "doc_type": t, "page": p,
           "source_uri": f"{GS}/{d}.pdf", "quote": q,
           "score": s} for d, t, p, s, q in _DOCS]
CITATION_FIELDS = ("chunk_id", "source_uri", "page", "quote",
                   "score")

# Document lengths, for the tool-chaining demos: retrieve
# finds chunks, and the cost tool bills whole documents.
DOC_PAGES = {"hr_policy_2026": 48, "msa_acme_2026": 32,
             "inv_2026_0412": 3, "gstr1_q1_fy27": 12,
             "rag_survey_2026": 24}          # 119 pages


def docs_of(citations: list) -> dict:
    """Distinct source documents behind a set of citations."""
    return {c["chunk_id"].split("#")[0]: True
            for c in citations}

def retrieve(query: str, doc_type: str = "all",
             top_k: int = 5) -> dict:
    """Retrieve grounded passages from DocuMind's corpus.

    Args:
        query: The question, in natural language
        doc_type: policy, contract, invoice, form,
            research_paper, or all
        top_k: How many passages to return
    """
    # A mock, but not a stub: it really filters and ranks, so
    # a question the corpus cannot answer returns NOTHING and
    # answerable=False. A mock that always succeeds teaches
    # that retrieval always succeeds - the one thing it never
    # does.
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [c for c in CORPUS
            if doc_type in ("all", c["doc_type"])
            and any(w in c["quote"].lower() for w in words)]
    hits.sort(key=lambda c: -c["score"])
    hits = hits[:top_k]
    top = hits[0]["score"] if hits else 0.0
    return {
        "citations": [{k: c[k] for k in CITATION_FIELDS}
                      for c in hits],
        "answerable": bool(hits),
        "confidence": ("high" if top >= 0.85 else
                       "medium" if hits else "low"),
    }

def calculate_processing_cost(
        total_pages: int, num_documents: int = 1,
        processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents
        num_documents: How many documents those pages span
        processing_type: standard, priority, or bulk
    """
    rate = RATES.get(processing_type, RATES["standard"])
    cost = total_pages * rate
    return {"num_documents": num_documents,
            "total_pages": total_pages,
            "processing_type": processing_type,
            "rate_per_page": rate,
            "cost_usd": round(cost, 2),
            "cost_inr": round(cost * USD_INR, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind pipeline usage statistics.

    Args:
        metric: queries, costs, latency, or users
        days: Number of days to look back
    """
    mock = {"queries": 1247, "costs": 18.50,
            "latency": 245, "users": 42}
    return {"metric": metric, "period": f"last {days} days",
            "value": mock.get(metric, 0), "trend": "+12%"}

TOOLS = [retrieve, calculate_processing_cost, get_usage_stats]
print(f"Defined {len(TOOLS)} DocuMind tools")

# The contract, demonstrated before any model sees it. Note
# the second call: the corpus cannot answer it, so answerable
# is False and citations is empty. That is a real answer, and
# it is what lesson 6.3 escalates to a web search on.
for q in ("What is the notice period for a senior engineer?",
          "What are the 2026 GDPR fines?"):
    out = retrieve(q)
    print(f"{out['answerable']!s:5} {out['confidence']:6} "
          f"{len(out['citations'])} citation(s)  {q[:38]}")

## Cell 2: Automatic Function Calling


In [ ]:
# SDK auto-executes functions
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='How much would it cost to process a 45-page contract?',
    config=types.GenerateContentConfig(tools=TOOLS)
)
print('=== Auto Function Calling ===')
print(response.text)


## Cell 3: Multiple Tool Selection


In [ ]:
# Test model selecting the right tool
queries = [
    'What does our refund policy say?',
    'How much to process 200 pages of invoices on the bulk tier?',
    'How many queries did we get last week?',
    'Hello, how are you today?',  # Should NOT call a function
]

for q in queries:
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=q,
        config=types.GenerateContentConfig(tools=TOOLS,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    )
    if r.function_calls:
        print(f'Q: {q}')
        print(f'  Tool: {r.function_calls[0].name}({r.function_calls[0].args})')
    else:
        print(f'Q: {q}')
        print(f'  Text: {r.text[:80]}...')
    print()


## Cell 4: Manual Loop (Without Auto-Execute)


In [ ]:
# Manual 4-step loop
from google.genai.types import FunctionDeclaration, Tool

# Step 1: Define tools manually
search_decl = FunctionDeclaration(
    name='retrieve',
    description='Search DocuMind documents by query.',
    parameters={
        'type': 'object',
        'properties': {
            'query': {'type': 'string', 'description': 'Search query'},
            'doc_type': {'type': 'string', 'enum': ['research_paper', 'invoice', 'legal', 'form', 'all']}
        },
        'required': ['query']
    }
)

tool = Tool(function_declarations=[search_decl])

# Step 2: Send prompt
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Find our refund policy documents',
    config=types.GenerateContentConfig(tools=[tool])
)

print('=== Manual Loop ===')
if response.function_calls:
    fc = response.function_calls[0]
    print(f'Function: {fc.name}')
    print(f'Args: {fc.args}')
    
    # Step 3: Execute
    result = retrieve(**fc.args)
    print(f'Result: {result}')
    
    # Step 4: Send result back
    final = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=[
            types.Content(role='user', parts=[
                types.Part.from_text(text='Find our refund policy documents')]),
            response.candidates[0].content,
            types.Content(role='user', parts=[
                types.Part.from_function_response(
                    name=fc.name, response=result)])
        ],
        config=types.GenerateContentConfig(tools=[tool])
    )
    print(f'\nFinal answer: {final.text}')


## Cell 5: Tool Config Modes


In [ ]:
# AUTO mode (default)
r_auto = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Hello!',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode='AUTO')))
)
print(f'AUTO + greeting: {r_auto.text[:80]}')
print(f'  Function calls: {len(r_auto.function_calls) if r_auto.function_calls else 0}')

# ANY mode (forced)
r_any = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Hello!',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode='ANY')))
)
print(f'\nANY + greeting: forced function call')
if r_any.function_calls:
    print(f'  Forced: {r_any.function_calls[0].name}({r_any.function_calls[0].args})')


## Cell 6: Chat Session with Tools


In [ ]:
# Multi-turn chat with function calling
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        system_instruction='You are DocuMind AI. Use tools to answer questions.'
    )
)

# Turn 1
r1 = chat.send_message('Find our refund policy')
print(f'Turn 1: {r1.text[:100]}...')

# Turn 2 (follow-up with context)
r2 = chat.send_message('How much would it cost to process that document?')
print(f'\nTurn 2: {r2.text[:100]}...')

# Turn 3 (different tool)
r3 = chat.send_message('What are our query stats for this week?')
print(f'\nTurn 3: {r3.text[:100]}...')

print(f'\nChat history: {len(chat.get_history())} messages')


## Cell 7: Error-Safe Dispatcher


In [ ]:
# Production-safe function dispatcher
def safe_execute(fc):
    ALLOWED = {
        'retrieve': retrieve,
        'calculate_processing_cost': calculate_processing_cost,
        'get_usage_stats': get_usage_stats,
    }
    DESTRUCTIVE = {'delete_document', 'send_email', 'modify_access'}
    
    if fc.name in DESTRUCTIVE:
        return {'error': f'Operation {fc.name} requires manual confirmation'}
    if fc.name not in ALLOWED:
        return {'error': f'Unknown function: {fc.name}'}
    try:
        return ALLOWED[fc.name](**fc.args)
    except Exception as e:
        return {'error': f'Execution failed: {str(e)}'}

# Test safe execution
print('Safe execute tests:')
for name, args in [('retrieve', {'query': 'test'}),
                   ('unknown_func', {}),
                   ('calculate_processing_cost', {'total_pages': 'abc'})]:
    class MockFC:
        pass
    fc = MockFC()
    fc.name = name
    fc.args = args
    result = safe_execute(fc)
    print(f'  {name}: {result}')


## ✅ Lesson 6.1 Complete!

- ✅ FunctionDeclaration with name, description, parameters
- ✅ Manual 4-step function calling loop
- ✅ Automatic function calling (SDK auto-executes)
- ✅ AUTO/ANY/NONE modes
- ✅ Multiple tool selection (model chooses)
- ✅ Parallel function calling
- ✅ Multi-turn chat with tools
- ✅ Error-safe dispatcher
- ✅ Security: never auto-execute destructive ops

**Next: Lesson 6.2 — Structured Tool Pipelines & Compositional Calling**
